# Coleta de Dados
## Ativo - ITUB4

In [1]:
# Importando as bibliotecas necessárias

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [2]:
# Definindo o ticker/ativo e a data de início para a coleta dos dados
TICKER = 'ITUB4.SA'
# Definindo a data de início (5 anos atrás a partir de hoje)
START_DATE = (datetime.now() - timedelta(days=5*365)).strftime('%Y-%m-%d')
#Definindo a data de término (hoje)
END_DATE = datetime.now().strftime('%Y-%m-%d')


In [3]:
# Realizando o download dos dados históricos do ativo utilizando a biblioteca yfinance do ativo ITUB4.SA, com os ajustes automáticos para dividendos e splits, e especificando o período de 5 anos
df = yf.download(TICKER, auto_adjust=True, start=START_DATE, end=END_DATE)

[*********************100%***********************]  1 of 1 completed


In [4]:
# Removendo o droplevel do nível "Ticker"
df.columns = df.columns.droplevel("Ticker")

In [5]:
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2021-05-11,18.025209,18.141169,17.857713,17.870598,28633629
2021-05-12,17.761089,17.909260,17.638688,17.844837,26098428
2021-05-13,18.276453,18.405297,17.761080,17.786849,48363464
2021-05-14,18.682310,18.798270,18.257127,18.463277,50573834
2021-05-17,18.669437,18.701647,18.276464,18.598573,42125619


In [6]:
df.tail()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-05-04,42.400002,43.250000,42.250000,43.110001,25336300
2026-05-05,42.459999,42.700001,42.119999,42.500000,29386700
2026-05-06,41.779999,43.290001,41.540001,43.240002,38718400
2026-05-07,40.790001,41.869999,40.520000,41.790001,36335300
2026-05-08,41.259998,41.549999,40.980000,41.160000,35920000


In [7]:
df.shape

(1246, 5)

In [8]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1246 entries, 2021-05-11 to 2026-05-08
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1246 non-null   float64
 1   High    1246 non-null   float64
 2   Low     1246 non-null   float64
 3   Open    1246 non-null   float64
 4   Volume  1246 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 58.4 KB


In [9]:
# Cálculo da Média Móvel Exponencial (EMA) de 60 dias
df['EMA_60'] = df['Close'].ewm(span=60, adjust=False).mean()

In [10]:
print(df[["Close", "EMA_60"]].head(65))
print(f"Valores nulos na EMA_60: {df['EMA_60'].isna().sum()}")

Price           Close     EMA_60
Date                            
2021-05-11  18.025209  18.025209
2021-05-12  17.761089  18.016550
2021-05-13  18.276453  18.025071
2021-05-14  18.682310  18.046620
2021-05-17  18.669437  18.067040
...               ...        ...
2021-08-05  19.387869  19.344923
2021-08-06  19.930723  19.364130
2021-08-09  20.163376  19.390335
2021-08-10  19.782089  19.403179
2021-08-11  20.001818  19.422807

[65 rows x 2 columns]
Valores nulos na EMA_60: 0


In [11]:
# Captando informações adicionais sobre o dataset, como valores ausentes, estatísticas descritivas e linhas com volume zerado
print("=== Valores ausentes ===")
print(df.isnull().sum())

print("\n=== Estatísticas descritivas ===")
print(df.describe())

print("\n=== Linhas com volume zerado ===")
zero_vol = df[df["Volume"] == 0]
print(f"Total: {len(zero_vol)}")
print(zero_vol)

=== Valores ausentes ===
Price
Close     0
High      0
Low       0
Open      0
Volume    0
EMA_60    0
dtype: int64

=== Estatísticas descritivas ===
Price        Close         High          Low         Open        Volume  \
count  1246.000000  1246.000000  1246.000000  1246.000000  1.246000e+03   
mean     24.058196    24.308724    23.807341    24.055424  3.281702e+07   
std       8.061812     8.126628     7.996774     8.064471  1.613884e+07   
min      13.783374    14.020221    13.757063    13.939864  6.695412e+06   
25%      17.819016    18.014622    17.639912    17.832713  2.197825e+07   
50%      21.196101    21.402212    20.892178    21.147197  2.941472e+07   
75%      27.635321    27.884135    27.382942    27.572046  4.022309e+07   
max      48.905392    49.202568    48.142637    48.588397  1.820390e+08   

Price       EMA_60  
count  1246.000000  
mean     23.456719  
std       7.350238  
min      15.163748  
25%      17.598304  
50%      19.551173  
75%      26.798663  
max   

In [12]:
df.head()

Price,Close,High,Low,Open,Volume,EMA_60
Date,,,,,,
2021-05-11,18.025209,18.141169,17.857713,17.870598,28633629,18.025209
2021-05-12,17.761089,17.909260,17.638688,17.844837,26098428,18.016550
2021-05-13,18.276453,18.405297,17.761080,17.786849,48363464,18.025071
2021-05-14,18.682310,18.798270,18.257127,18.463277,50573834,18.046620
2021-05-17,18.669437,18.701647,18.276464,18.598573,42125619,18.067040


In [13]:
# Removendo linhas com volume zerado
df = df[df["Volume"] > 0]
print(f"Dataset após remoção de linhas com volume zerado: {df.shape}")

Dataset após remoção de linhas com volume zerado: (1246, 6)


In [14]:
#Gerando arquivo CSV do dataset processado
df.to_csv("/Users/lucaszaninidasilva/Mackenzie/SetimoSemestre/IA/PROJETO/src/database/processed/ITUB4_processed.csv", index=True)